# Solution Postprocessing

In [ ]:
# Imports
import os
import pandas as pd
import re
import shutil
import numpy as np

In [12]:
def extract(pattern, as_int=False):
    match = re.search(pattern, text)
    if not match:
        return None
    value_str = re.sub(r"[^0-9eE\+\-\.]", "", match.group(1).strip())
    try:
        if as_int:
            return int(float(value_str))  # Convert via float first for safety
        return float(value_str)
    except ValueError:
        return None

In [72]:
base_model_dir = "2_Model_and_Solution"  # Root directory containing harvest schedules

for hsched_name in os.listdir(base_model_dir):
    hsched_path = os.path.join(base_model_dir, hsched_name)
    if not os.path.isdir(hsched_path):
        continue

    print(f"📂 Processing harvest schedule: {hsched_name}")

    # Store per-schedule results here
    hsched_results = []

    for component in os.listdir(hsched_path):
        comp_path = os.path.join(hsched_path, component)
        summary_path = os.path.join(comp_path, "solution_summary.txt")
        nonzero_path = os.path.join(comp_path, "solution_nonzero.csv")

        if not os.path.exists(summary_path):
            continue

        # --- Parse solver summary
        with open(summary_path, "r") as f:
            text = f.read()

        def extract(pattern, as_int=False):
            match = re.search(pattern, text)
            if not match:
                return None
            value_str = re.sub(r"[^0-9eE\+\-\.]", "", match.group(1).strip())
            try:
                return int(float(value_str)) if as_int else float(value_str)
            except ValueError:
                return None

        ub = extract(r"Objective value \(UB\): ([\d\.\-eE]+)")
        lb = extract(r"Best bound \(LB\): ([\d\.\-eE]+)")
        gap = extract(r"Relative MIP gap: ([\d\.\-eE]+)")
        n_vars = extract(r"Total variables: (\d+)", as_int=True)
        nonzero_vars = extract(r"Non-zero variables: (\d+)", as_int=True)
        runtime = None

        # --- Load nonzero variable data
        if os.path.exists(nonzero_path):
            df_sol = pd.read_csv(nonzero_path)

            if "variable" not in df_sol.columns:
                print(f"⚠️ No 'variable' column in {component}, skipping.")
                continue

            # Extract variable components
            df_sol["_prefix"] = df_sol["variable"].astype(str).str.extract(r"^([CMU])")
            df_sol["_width"] = df_sol["variable"].astype(str).str.extract(r"_(timber|wide)_")
            df_sol["_period"] = df_sol["variable"].astype(str).str.extract(r"t?(\d)$")

            # --- Totals across all periods
            total_construct_timber = int(((df_sol["_prefix"] == "C") & (df_sol["_width"] == "timber")).sum())
            total_construct_wide   = int(((df_sol["_prefix"] == "C") & (df_sol["_width"] == "wide")).sum())
            total_maintain_timber  = int(((df_sol["_prefix"] == "M") & (df_sol["_width"] == "timber")).sum())
            total_maintain_wide    = int(((df_sol["_prefix"] == "M") & (df_sol["_width"] == "wide")).sum())
            total_upgrade          = int((df_sol["_prefix"] == "U").sum())

            # Define all possible periods (0 excluded)
            all_periods = ["1", "2", "3", "4", "5"]
            all_combos = ["C_timber", "C_wide", "M_timber", "M_wide", "U_"]

            # Create _combo as before
            df_sol["_combo"] = (
                df_sol["_prefix"].astype(str) + "_" + df_sol["_width"].fillna("")  # U has no width
            )

            # Pivot and fill missing combinations with 0
            per_period_counts = (
                df_sol.groupby(["_period", "_combo"]).size()
                .unstack(fill_value=0)
                .reindex(index=all_periods, columns=all_combos, fill_value=0)
            )

            # Flatten into dict
            period_dict = {}
            for period in per_period_counts.index:
                for combo in per_period_counts.columns:
                    count = int(per_period_counts.loc[period, combo] or 0)
                    prefix, width = (combo.split("_") + [""])[:2]
                    width_suffix = f"_{width}" if width else ""
                    col_name = ""
                    if prefix == "C":
                        col_name = f"constructed{width_suffix}_t{period}"
                    elif prefix == "M":
                        col_name = f"maintained{width_suffix}_t{period}"
                    elif prefix == "U":
                        col_name = f"upgraded_t{period}"
                    period_dict[col_name] = count

        # --- Determine solver status
        if gap is not None:
            if gap < 1e-6:  # effectively zero
                solver_status = "Optimal"
            else:
                solver_status = f"Gap"
        else:
            solver_status = "Unknown"

        # --- Add component-level summary
        hsched_results.append({
            "harvest_schedule": hsched_name,
            "component": component,
            "objective_value": round(ub,2),
            "best_bound": round(lb,2),
            "solver_status": solver_status,
            "mip_rel_gap": gap,
            "total_vars": int(n_vars) if n_vars is not None else 0,
            "nonzero_vars": int(nonzero_vars) if nonzero_vars is not None else 0,
            "constructed_timber": total_construct_timber,
            "constructed_wide": total_construct_wide,
            "maintained_timber": total_maintain_timber,
            "maintained_wide": total_maintain_wide,
            "upgraded": total_upgrade,
            **period_dict,
            "runtime_sec": runtime
        })

    # --- Save per-harvest-schedule summary
    if hsched_results:
        summary_df = pd.DataFrame(hsched_results)

        # --- Compute totals from per-period columns
        summary_df['constructed_timber_total'] = summary_df[[f'constructed_timber_t{i}' for i in range(1,6)]].sum(axis=1)
        summary_df['constructed_wide_total'] = summary_df[[f'constructed_wide_t{i}' for i in range(1,6)]].sum(axis=1)
        summary_df['maintained_timber_total'] = summary_df[[f'maintained_timber_t{i}' for i in range(1,6)]].sum(axis=1)
        summary_df['maintained_wide_total'] = summary_df[[f'maintained_wide_t{i}' for i in range(1,6)]].sum(axis=1)
        summary_df['upgraded_total'] = summary_df[[f'upgraded_t{i}' for i in range(1,6)]].sum(axis=1)

        # --- Verify totals match original totals
        totals_match = (
            (summary_df['constructed_timber_total'] == summary_df['constructed_timber']) &
            (summary_df['constructed_wide_total'] == summary_df['constructed_wide']) &
            (summary_df['maintained_timber_total'] == summary_df['maintained_timber']) &
            (summary_df['maintained_wide_total'] == summary_df['maintained_wide']) &
            (summary_df['upgraded_total'] == summary_df['upgraded'])
        )

        if totals_match.all():
            print("✅ All period sum-up match the totals.")
        else:
            print("⚠️ Some totals do not match the period sum-up!")

        # --- Drop original total columns (keep per-period counts)
        original_totals_cols = ['constructed_timber', 'constructed_wide', 'maintained_timber', 'maintained_wide', 'upgraded']
        summary_df = summary_df.drop(columns=original_totals_cols)

        # --- Optional: reorder columns
        main_cols = [
            "harvest_schedule",
            "component", "objective_value", "solver_status", "best_bound", "mip_rel_gap",
            "total_vars", "nonzero_vars",
            "runtime_sec"
        ]
        # Per-period totals and counts
        total_cols = ['constructed_timber_total', 'constructed_wide_total',
                    'maintained_timber_total', 'maintained_wide_total', 'upgraded_total']
        period_cols = [c for c in summary_df.columns if re.search(r'_t[1-5]$', c)]
        other_cols = [c for c in summary_df.columns if c not in main_cols + total_cols + period_cols]

        summary_df = summary_df[main_cols + total_cols + sorted(period_cols) + sorted(other_cols)]

        out_path = os.path.join(hsched_path, f"{hsched_name}_solution_summary.csv")
        summary_df.to_csv(out_path, index=False)
        print(f"✅ Saved summary for {hsched_name} → {out_path}")
    else:
        print(f"⚠️ No valid components found in {hsched_name}, skipping summary.")


📂 Processing harvest schedule: mres
✅ All period sum-up match the totals.
✅ Saved summary for mres → 2_Model_and_Solution\mres\mres_solution_summary.csv
📂 Processing harvest schedule: mwood
✅ All period sum-up match the totals.
✅ Saved summary for mwood → 2_Model_and_Solution\mwood\mwood_solution_summary.csv
📂 Processing harvest schedule: stake
✅ All period sum-up match the totals.
✅ Saved summary for stake → 2_Model_and_Solution\stake\stake_solution_summary.csv
📂 Processing harvest schedule: summary_results_after_1strun_10min
⚠️ No valid components found in summary_results_after_1strun_10min, skipping summary.
📂 Processing harvest schedule: summary_results_after_2ndrun_180min
⚠️ No valid components found in summary_results_after_2ndrun_180min, skipping summary.


## load 3 summaries for the 3 schedules

In [90]:
dfs = pd.read_csv(f'{base_model_dir}/stake_solution_summary_all_components.csv').reset_index(drop=True)
dfs

,harvesting_schedule,component_name,UB,LB,MIP_gap,solver_runtime_seconds,total_variables,nonzero_variables
0,stake,comp_16,16309.13,16307.666991,0.00009,442.97,14150,399


In [80]:
dfmr = pd.read_csv(f'{base_model_dir}/mres_solution_summary_all_components.csv').reset_index(drop=True)
dfmr

,harvesting_schedule,component_name,UB,LB,MIP_gap,solver_runtime_seconds,total_variables,nonzero_variables
0,mres,comp_14_15_48_merged,56371.90,51501.998176,0.086389,4201.40,50950,1524
1,mres,comp_5_39_merged,129943.01,75492.413252,0.419034,4200.65,72250,2754


In [81]:
dfmw = pd.read_csv(f'{base_model_dir}/mwood_solution_summary_all_components.csv').reset_index(drop=True)
dfmw

,harvesting_schedule,component_name,UB,LB,MIP_gap,solver_runtime_seconds,total_variables,nonzero_variables
0,mwood,comp_14_15_48_merged,58344.65,52393.362442,0.102002,4200.29,50950,1749
1,mwood,comp_5_39_merged,123589.02,78008.571008,0.368807,4200.37,72250,3028


## calculate core metrics

In [82]:
df = dfmr

In [83]:
# Step 1: Core global metrics
total_UB = df["UB"].sum()
total_LB = round(df["LB"].sum(),2)
aggregated_gap = round((100*(total_UB - total_LB) / total_UB),2).astype(str)+ "%"
mean_runtime = round(df["solver_runtime_seconds"].mean(),2)

# Step 2: Component contributions (% of total UB)
df["UB_pct"] = df["UB"] / total_UB * 100

# Step 3: Create global metrics table
global_metrics = pd.DataFrame({
    "Total_UB": [total_UB],
    "Total_LB": [total_LB],
    "Aggregated_MIP_Gap": [aggregated_gap],
    "Mean_Runtime_sec": [mean_runtime]
})

# Step 4: Display results
print("=== Global Metrics ===")
print(global_metrics.to_string(index=False))

=== Global Metrics ===
 Total_UB  Total_LB Aggregated_MIP_Gap  Mean_Runtime_sec
186314.91 126994.41             31.84%           4201.02


In [84]:
df = dfmw

In [85]:
# Step 1: Core global metrics
total_UB = df["UB"].sum()
total_LB = round(df["LB"].sum(),2)
aggregated_gap = round((100*(total_UB - total_LB) / total_UB),2).astype(str)+ "%"
mean_runtime = round(df["solver_runtime_seconds"].mean(),2)

# Step 2: Component contributions (% of total UB)
df["UB_pct"] = df["UB"] / total_UB * 100

# Step 3: Create global metrics table
global_metrics = pd.DataFrame({
    "Total_UB": [total_UB],
    "Total_LB": [total_LB],
    "Aggregated_MIP_Gap": [aggregated_gap],
    "Mean_Runtime_sec": [mean_runtime]
})

# Step 4: Display results
print("=== Global Metrics ===")
print(global_metrics.to_string(index=False))

=== Global Metrics ===
 Total_UB  Total_LB Aggregated_MIP_Gap  Mean_Runtime_sec
181933.67 130401.93             28.32%           4200.33


In [86]:
df = dfs 

In [87]:
# Step 1: Core global metrics
total_UB = df["UB"].sum()
total_LB = round(df["LB"].sum(),2)
aggregated_gap = round((100*(total_UB - total_LB) / total_UB),2).astype(str)+ "%"
mean_runtime = round(df["solver_runtime_seconds"].mean(),2)

# Step 2: Component contributions (% of total UB)
df["UB_pct"] = df["UB"] / total_UB * 100

# Step 3: Create global metrics table
global_metrics = pd.DataFrame({
    "Total_UB": [total_UB],
    "Total_LB": [total_LB],
    "Aggregated_MIP_Gap": [aggregated_gap],
    "Mean_Runtime_sec": [mean_runtime]
})

# Step 4: Display results
print("=== Global Metrics ===")
print(global_metrics.to_string(index=False))

=== Global Metrics ===
 Total_UB  Total_LB Aggregated_MIP_Gap  Mean_Runtime_sec
186258.59 134262.68             27.92%           4200.34


## create tables for thesis

In [88]:
# Round numeric columns
df['LB'] = df['LB'].round(2)
df['MIP_gap'] = df['MIP_gap'].round(4)

# Convert MIP gap to percentage string
df['MIP_gap'] = (100 * df['MIP_gap']).round(2)

# Replace 0.0 with "-" and format the rest as percentages
df['MIP_gap'] = df['MIP_gap'].apply(lambda x: '-' if x == 0.0 else f'{x:.2f}%')
df.drop(columns=["harvesting_schedule"], inplace=True)

# Extract numeric part of component_name for sorting
df['comp_num'] = df['component_name'].apply(lambda x: int(re.findall(r'\d+', x)[0]))

# Sort by this numeric column
df = df.sort_values(by='comp_num').reset_index(drop=True)

# Drop the helper column if you want
df = df.drop(columns='comp_num')
df

,component_name,UB,LB,MIP_gap,solver_runtime_seconds,total_variables,nonzero_variables,UB_pct
0,comp_5_39_merged,127913.94,82470.80,35.53%,4200.40,72250,3015,68.675458
1,comp_14_15_48_merged,58344.65,51791.89,11.23%,4200.27,50950,1786,31.324542


## other

In [17]:
# --- Compute Global Metrics ---
total_obj = df["UB"].sum()
total_lb = df["LB"].sum()
aggr_mip_gap = (total_obj - total_lb) / total_obj if total_obj != 0 else np.nan
mean_obj = df["UB"].mean()
median_obj = df["UB"].median()
total_runtime = df["solver_runtime_seconds"].sum()
mean_runtime = df["solver_runtime_seconds"].mean()

# --- Compute Component Contributions ---
df["component_contribution_pct"] = 100 * df["UB"] / total_obj if total_obj != 0 else 0
df_sorted = df.sort_values("component_contribution_pct", ascending=False)

# --- Save Metrics ---
hsched_name = df["harvesting_schedule"].iloc[0]
out_dir = "."  # change this to your output directory if needed

core_metrics_path = os.path.join(out_dir, f"{hsched_name}_core_metrics.txt")
with open(core_metrics_path, "w") as f:
    f.write(f"Harvest Schedule: {hsched_name}\n")
    f.write(f"Total Objective Value (ΣUB): {round(total_obj, 2)}\n")
    f.write(f"Total Lower Bound (ΣLB): {round(total_lb, 2)}\n")
    f.write(f"Aggregated MIP Gap: {round(aggr_mip_gap, 4)}\n")
    f.write(f"Mean Objective Value: {round(mean_obj, 2)}\n")
    f.write(f"Median Objective Value: {round(median_obj, 2)}\n")
    f.write(f"Total Runtime (sec): {round(total_runtime, 2)}\n")
    f.write(f"Mean Runtime (sec): {round(mean_runtime, 2)}\n")
print(f"✅ Saved global core metrics → {core_metrics_path}")

# --- Save Component Importance ---
comp_imp_path = os.path.join(out_dir, f"{hsched_name}_component_importance.csv")
df_sorted.to_csv(comp_imp_path, index=False)
print(f"🔥 Saved component importance CSV → {comp_imp_path}")

# --- Save Top Components ---
top_n = 10
top_txt_path = os.path.join(out_dir, f"{hsched_name}_top_components.txt")
with open(top_txt_path, "w") as f:
    f.write(f"Harvest Schedule: {hsched_name}\n")
    f.write("Top Components by % Contribution to Total Objective:\n")
    for i, row in df_sorted.head(top_n).iterrows():
        f.write(f"{i+1:02d}. {row['component_name']}: {round(row['UB'],2)} ({round(row['component_contribution_pct'],2)}%)\n")
print(f"🏆 Saved top {top_n} components → {top_txt_path}")

# --- Print Summary to Console ---
print("\n📊 Global Metrics Summary:")
print(f"Total Objective Value: {total_obj:,.2f}")
print(f"Total Lower Bound: {total_lb:,.2f}")
print(f"Aggregated MIP Gap: {aggr_mip_gap:.4f}")
print(f"Mean Objective Value: {mean_obj:,.2f}")
print(f"Median Objective Value: {median_obj:,.2f}")
print(f"Total Runtime (sec): {total_runtime:,.2f}")
print(f"Mean Runtime (sec): {mean_runtime:,.2f}")# --- Compute Global Metrics ---
total_obj = df["UB"].sum()
total_lb = df["LB"].sum()
aggr_mip_gap = (total_obj - total_lb) / total_obj if total_obj != 0 else np.nan
mean_obj = df["UB"].mean()
median_obj = df["UB"].median()
total_runtime = df["solver_runtime_seconds"].sum()
mean_runtime = df["solver_runtime_seconds"].mean()

# --- Compute Component Contributions ---
df["component_contribution_pct"] = 100 * df["UB"] / total_obj if total_obj != 0 else 0
df_sorted = df.sort_values("component_contribution_pct", ascending=False)

# --- Save Metrics ---
hsched_name = df["harvesting_schedule"].iloc[0]
out_dir = "."  # change this to your output directory if needed

core_metrics_path = os.path.join(out_dir, f"{hsched_name}_core_metrics.txt")
with open(core_metrics_path, "w") as f:
    f.write(f"Harvest Schedule: {hsched_name}\n")
    f.write(f"Total Objective Value (ΣUB): {round(total_obj, 2)}\n")
    f.write(f"Total Lower Bound (ΣLB): {round(total_lb, 2)}\n")
    f.write(f"Aggregated MIP Gap: {round(aggr_mip_gap, 4)}\n")
    f.write(f"Mean Objective Value: {round(mean_obj, 2)}\n")
    f.write(f"Median Objective Value: {round(median_obj, 2)}\n")
    f.write(f"Total Runtime (sec): {round(total_runtime, 2)}\n")
    f.write(f"Mean Runtime (sec): {round(mean_runtime, 2)}\n")
print(f"✅ Saved global core metrics → {core_metrics_path}")

# --- Save Component Importance ---
comp_imp_path = os.path.join(out_dir, f"{hsched_name}_component_importance.csv")
df_sorted.to_csv(comp_imp_path, index=False)
print(f"🔥 Saved component importance CSV → {comp_imp_path}")

# --- Save Top Components ---
top_n = 10
top_txt_path = os.path.join(out_dir, f"{hsched_name}_top_components.txt")
with open(top_txt_path, "w") as f:
    f.write(f"Harvest Schedule: {hsched_name}\n")
    f.write("Top Components by % Contribution to Total Objective:\n")
    for i, row in df_sorted.head(top_n).iterrows():
        f.write(f"{i+1:02d}. {row['component_name']}: {round(row['UB'],2)} ({round(row['component_contribution_pct'],2)}%)\n")
print(f"🏆 Saved top {top_n} components → {top_txt_path}")

# --- Print Summary to Console ---
print("\n📊 Global Metrics Summary:")
print(f"Total Objective Value: {total_obj:,.2f}")
print(f"Total Lower Bound: {total_lb:,.2f}")
print(f"Aggregated MIP Gap: {aggr_mip_gap:.4f}")
print(f"Mean Objective Value: {mean_obj:,.2f}")
print(f"Median Objective Value: {median_obj:,.2f}")
print(f"Total Runtime (sec): {total_runtime:,.2f}")
print(f"Mean Runtime (sec): {mean_runtime:,.2f}")

UnicodeEncodeError: 'charmap' codec can't encode character '\u03a3' in position 23: character maps to <undefined>

In [4]:
summary_df.head()

,harvest_schedule,component,objective_value,solver_status,best_bound,mip_rel_gap,total_vars,nonzero_vars,runtime_sec,constructed_timber_total,...,maintained_wide_t1,maintained_wide_t2,maintained_wide_t3,maintained_wide_t4,maintained_wide_t5,upgraded_t1,upgraded_t2,upgraded_t3,upgraded_t4,upgraded_t5
0,mwood,comp_1,1552.36,Optimal,1552.36,0.000000,4900,83,None,19,...,0,2,2,3,3,0,0,1,0,6
1,mwood,comp_10,19597.69,Gap,19595.85,0.000094,14350,400,None,44,...,0,15,24,30,32,0,7,5,6,6
2,mwood,comp_14_15_48_merged,61299.85,Gap,50984.06,0.168284,50950,1759,None,249,...,0,36,53,85,131,0,19,29,53,41
3,mwood,comp_16,16309.13,Gap,16307.52,0.000098,14150,399,None,74,...,0,1,2,7,11,0,1,1,4,15
4,mwood,comp_18,2279.64,Optimal,2279.64,0.000000,1600,85,None,18,...,0,0,0,0,4,0,0,0,4,4


In [6]:
base_model_dir = "2_Model_and_Solution"
output_file = os.path.join(base_model_dir, "all_harvest_schedules_summary.csv")

# Collect all CSV summary files from subfolders
csv_files = []
for subfolder in os.listdir(base_model_dir):
    subfolder_path = os.path.join(base_model_dir, subfolder)
    if not os.path.isdir(subfolder_path):
        continue
    for f in os.listdir(subfolder_path):
        if f.endswith("_solution_summary.csv"):
            csv_files.append(os.path.join(subfolder_path, f))

if not csv_files:
    print("⚠️ No summary CSV files found in subfolders.")
else:
    # Read and concatenate all summaries
    df_list = [pd.read_csv(f) for f in csv_files]
    consolidated_df = pd.concat(df_list, ignore_index=True)

    # Optional: sort by harvest schedule and component
    consolidated_df = consolidated_df.sort_values(by=["harvest_schedule", "component"])

    # Save consolidated summary
    consolidated_df.to_csv(output_file, index=False)
    print(f"✅ Consolidated summary saved to {output_file}")

✅ Consolidated summary saved to 2_Model_and_Solution\all_harvest_schedules_summary.csv


# save summary results before rerun extra folder

In [ ]:
def collect_results(main_dir, nrun, nmin):
    # Define subdirectories to search
    subdirs = ["mres", "mwood", "stake"]
    
    # Define the output folder path
    output_dir = os.path.join(main_dir, f"summary_results_after_{nrun}run_{nmin}min")
    os.makedirs(output_dir, exist_ok=True)

    # Collect from main directory + each subdirectory
    search_dirs = [main_dir] + [os.path.join(main_dir, s) for s in subdirs]

    for directory in search_dirs:
        if not os.path.exists(directory):
            print(f"⚠️ Skipping missing directory: {directory}")
            continue

        for file in os.listdir(directory):
            if file.endswith(".csv") or file.endswith(".txt"):
                src_path = os.path.join(directory, file)
                dest_path = os.path.join(output_dir, file)

                # Handle name conflicts by adding a counter
                base, ext = os.path.splitext(file)
                counter = 1
                while os.path.exists(dest_path):
                    dest_path = os.path.join(output_dir, f"{base}_{counter}{ext}")
                    counter += 1

                shutil.copy2(src_path, dest_path)

    print(f"✅ All CSV and TXT files have been copied to:\n{output_dir}")

# Example usage:
collect_results("2_Model_and_Solution", nrun='re', nmin=240)

✅ All CSV and TXT files have been copied to:
2_Model_and_Solution\summary_results_after_3rdrun_240min
